# Silver Agent — Data & Output Tests
Quick sanity checks for fetchers, signals, scores, and PDF generation.
Run cells top-to-bottom; each section is independent after the setup cell.

In [ ]:
# ── Setup: add project root to path so all modules resolve ──────────────────
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(ROOT, ".env"))

print("Root:", ROOT)
print("Path OK")

## 1. Price Fetchers

In [ ]:
from src.fetchers.price import (
    fetch_silver_price, fetch_gold_price,
    fetch_dxy_price, fetch_us10y_price,
)

silver = fetch_silver_price()
gold   = fetch_gold_price()
dxy    = fetch_dxy_price()
us10y  = fetch_us10y_price()

print(f"Silver : ${silver['price']:.2f}  ({silver['change_pct']:+.2f}%)")
print(f"Gold   : ${gold['price']:.2f}  ({gold['change_pct']:+.2f}%)")
print(f"DXY    : {dxy['price']:.2f}  ({dxy['change_pct']:+.2f}%)")
print(f"US10Y  : {us10y['price']:.3f}%  ({us10y['change_pct']:+.2f}%)")

assert silver['price'] > 0
assert gold['price'] > 0
print("\n[PASS] All prices fetched and positive")

## 2. 30-Day History & RSI

In [ ]:
from src.fetchers.price import fetch_silver_history

history = fetch_silver_history(30)
closes  = [h['close'] for h in history]

print(f"Days returned : {len(history)}")
print(f"Price range   : ${min(closes):.2f} - ${max(closes):.2f}")
print(f"Latest close  : ${closes[-1]:.2f}")

assert len(history) >= 10, "Too few history rows"
print("\n[PASS] History looks reasonable")

## 3. Signals Computation

In [ ]:
from src.analysis.signals import compute_price_signals, compute_data_quality, format_signals_for_prompt

signals      = compute_price_signals(silver, gold, dxy, us10y, history)
data_quality = compute_data_quality(silver, gold, dxy, us10y)
signals_text = format_signals_for_prompt(signals, data_quality)

s = signals['silver']
print(f"RSI-14      : {s['rsi_14']}  -> {s['signal']}")
print(f"Volatility  : {s['volatility_30d']}%")
print(f"vs 30d high : {s['vs_30d_high_pct']:+.1f}%")
print(f"Sig. move   : {s['significant_move']}")
print(f"G/S Ratio   : {signals['ratio']['value']}  ({signals['ratio']['signal']})")
print(f"Reliability : {data_quality['reliability']}")
print()
print(signals_text)

assert signals['ratio']['value'] > 0
assert data_quality['reliability'] in ('LOW', 'MEDIUM', 'HIGH')
print("\n[PASS] Signals computed OK")

## 4. News Fetcher

In [ ]:
from src.fetchers.news import fetch_articles

articles = fetch_articles()
print(f"Articles fetched: {len(articles)}")

for i, a in enumerate(articles[:3], 1):
    print(f"\n{i}. {a['title']}")
    print(f"   {a.get('date', 'N/A')}  |  {a.get('url', '')[:60]}")

assert len(articles) > 0, "No articles returned"
print("\n[PASS] News fetcher OK")

## 5. Score Extraction from Saved Briefing

In [ ]:
import json, pathlib
from src.agents.summarizer import extract_scores

daily_dir = pathlib.Path(ROOT) / "outputs" / "daily"
latest    = sorted(daily_dir.iterdir())[-1]
briefing_file = latest / "briefing" / "briefing.txt"
scores_file   = latest / "briefing" / "scores.json"

raw_briefing = briefing_file.read_text()
saved_scores = json.loads(scores_file.read_text())

print(f"Loaded: {briefing_file}")
print(f"Length: {len(raw_briefing)} chars")
print(f"Saved scores: {saved_scores}")

_clean, reparsed = extract_scores(raw_briefing)
print(f"\nRe-parsed: {reparsed}")
print(f"Overall  : {reparsed.get('overall', '?')}/10")
print(f"Verdict  : {reparsed.get('verdict', 'N/A')}")

print("\n[PASS] Score extraction OK")

## 6. PDF Generation Smoke Test

In [ ]:
import json, pathlib
from src.dashboard.app import generate_pdf

daily_dir = pathlib.Path(ROOT) / "outputs" / "daily"
latest    = sorted(daily_dir.iterdir())[-1]

prices   = json.loads((latest / "raw" / "prices.json").read_text())
scores   = json.loads((latest / "briefing" / "scores.json").read_text())
raw_sig  = json.loads((latest / "raw" / "signals.json").read_text())
briefing = (latest / "briefing" / "briefing.txt").read_text()

dxy_stub   = {"price": raw_sig["dxy"]["value"],   "change_pct": raw_sig["dxy"]["change_pct"]}
us10y_stub = {"price": raw_sig["us10y"]["value"], "change_pct": raw_sig["us10y"]["change_pct"]}

pdf_bytes = generate_pdf(
    briefing      = briefing,
    silver        = prices["silver"],
    gold          = prices["gold"],
    briefing_date = str(latest.name),
    scores        = scores,
    dxy           = dxy_stub,
    us10y         = us10y_stub,
)

out_path = pathlib.Path(ROOT) / "outputs" / f"test_pdf_{latest.name}.pdf"
out_path.write_bytes(pdf_bytes)

print(f"PDF size : {len(pdf_bytes):,} bytes")
print(f"Saved to : {out_path}")

assert len(pdf_bytes) > 5_000, "PDF suspiciously small"
assert pdf_bytes[:4] == b'%PDF', "Not a valid PDF header"
print("\n[PASS] PDF generated OK")